In [4]:
import numpy as np
import time
import sys
sys.path.append('../ExtraFiles/')
sys.path.append('../StarModels/')
sys.path.append('../accretors/')
sys.path.append('../')

import matplotlib.pyplot as plt
import CEconstants  as const
import CEParameters as cp
import starModels      as sm
import accretorModel
import InspiralParameters as ip
import EnergyHelper as eh
import EnergyInjections as ej
import CEdragModels as cedm
import CEjetModels  as cejm
import MESAGen      as mg
import InlistHelper as ih
import CEIO         as ceio
import CE_Terminate as cet

In [5]:
#test = mg.readModfile("12.0MsunInput.mod")
#rho = np.exp(ih.changeDEModfile(test['lnd'], 'D', 'E'))
#R = np.exp(ih.changeDEModfile(test['lnR'], 'D', 'E'))
#T = np.exp(ih.changeDEModfile(test['lnT'], 'D', 'E'))
#arr = T
#violations = [i for i in range(len(arr)-1) if arr[i] >= arr[i+1]]
#print("Violations at indices:", violations)
#for i in violations:
#    print(f"arr[{i}] = {arr[i]} >= arr[{i+1}] = {arr[i+1]}")
#fig, ax = plt.subplots()
#ax.loglog(R, T)
#ax.loglog(R, rho)
#ax.axvline(R[401])
#ax.axvline(R[412])

#fig, ax = plt.subplots()
#ax.loglog(R, rho)
#ax.loglog(R, rho)
#ax.axvline(R[401])
#ax.axvline(R[412])

In [6]:
pyMesaContainer = mg.pyMesa_context('../pyMesaStarCache/')
runParameters = cp.RunTimeParameters()

eos.inlist
eos.inlist contents:
&eos
  ! Use all the standard EOS sources (FreeEOS, OPAL, HELM)
  !use_FreeEOS       = .true.
  !use_OPAL_SCVH     = .true.
  !use_Skye          = .true.
  !use_PC            = .false.

  ! Tweak FreeEOS limits to ensure wide coverage
  !logT_min_FreeEOS_lo = 3.0
  !logT_min_FreeEOS_hi = 3.1
  !logT_max_FreeEOS_lo = 8.1
  !logT_max_FreeEOS_hi = 8.2

  !logRho_min_FreeEOS_lo = -18.1
  !logRho_min_FreeEOS_hi = -18.0
  !logRho_max_FreeEOS_lo = 10.0
  !logRho_max_FreeEOS_hi = 10.1

  ! Allow HELM to take over gracefully if FreeEOS fails
  !logT_all_HELM = 7.6
  !logT_low_all_HELM = 2.3

  ! Prevent crashes on EOS failure
  !okay_to_convert_ierr_to_skip = .true.
/

Loaded Modules for eos, const, math, chem...


In [7]:
donorStar       = sm.donorStar(runParameters.starModelPath + runParameters.starProfileModel)
donorModData     = mg.readModfile(runParameters.starModelPath + runParameters.starModModel)
accretorStar    = accretorModel.accretorStar(runParameters)
accretorStar.loadEOS()
simParameters   = ip.InspiralParameters(runParameters, donorStar, accretorStar)

ih.initInspiralFiles(runParameters, donorStar)
ih.initializeMesaInlist(runParameters, simParameters, donorModData)
ceio.writeInspiralParamters(runParameters, simParameters, donorStar)

'NSAccretor1' already exists within '../..'.
'RSGDonor1' already exists within '../..'.
Finished Creating Accretor and Donor star save directories...
Finished copying inital MESA mod and profile files...
MESA reading inlist_project inlist file..
MESA inlist file and path names set...
MESA inlist current age set to match MESA star profile...


In [8]:
def mainSimulation(runParameters, simParameters, donorStar, accretorStar, pyMesaObject, donorModData):

    # Save initial conditions
    ceio.saveCEData(simParameters, runParameters)

    while(simParameters.runFlag):
        
        itertime = time.time()
        donorStar.readMesaProfile(runParameters.profile)
        dt = runParameters.dtType
        simParameters.t2 = simParameters.t1 + dt
        
        # Call EOS
        # 1. Calculate Accretion Radius ranges
        simParameters.calculateDragRadius(runParameters)
        simParameters.calculateJetRadius(runParameters, donorStar)
        
        # 2. Calculate drag/jet forces + inspiral evolution
        eom(runParameters, simParameters, donorStar, accretorStar)

        # Integration Step
        simParameters.m1 = simParameters.m1 + dt*accretorStar.dm_dt
        simParameters.a  = simParameters.a  + dt*simParameters.da_dt
        simParameters.th = simParameters.th + dt*simParameters.th_dt
        simParameters.Om = simParameters.Om + dt*accretorStar.dOm_dt

        # CALLING MESA
        finalModData = ej.updateRunMesa(pyMesaObject, simParameters, runParameters, donorStar, donorModData)

        # check termination
        cet.checkTermination(runParameters, simParameters)

        # Save Stuff..
        ceio.printSimulationOutput(runParameters, simParameters, donorStar)
        ceio.saveCEData(simParameters, runParameters)
        ceio.saveInspiralFile(runParameters.profile, runParameters.pathDonor, 
                              simParameters.savenum, '_')    # Save Mesa profile
        ceio.checkPointSave(runParameters, simParameters)
        
        simParameters.savenum += 1
        simParameters.count   += 1        
        simParameters.t1     = simParameters.t2
        donorModData         = finalModData
        simParameters.wallclock += time.time()-itertime

    # this a final saved run in case we finish and want to restart it to go further...
    ceio.restartPointSave(runParameters)
    # End of Run ...
    return

In [9]:
def eom(runParameters, simParameters, donorStar, accretorStar):
    
    simParameters.getLocalVariable(donorStar)

    rMin   = runParameters.accretorRadi
    jetEff = runParameters.jetEff
    freeCd = runParameters.freeCd
    dt     = runParameters.dtType
    a      = simParameters.a
    Rs     = donorStar.R
    Rhos   = donorStar.rho
    rho_   = simParameters.rho
    Ra     = simParameters.Ra
    Eorb   = simParameters.Eorb
    m1     = accretorStar.m
    P      = simParameters.P

    accretorStar.getAccretorValues(simParameters)
    
    # TODO: Double check i am calculating the difference gradient correctly.
    simParameters.epGrad = eh.getEPGrad(Rhos, Rs, a, Ra, rho_, simParameters)
    
    # Drag + Jet Method
    # TODO: Check this is done properly
    dMdot, Fd, dEdrag = cedm.chooseDragMethod(runParameters, simParameters)
    dEjet             = cejm.chooseJetMethod(runParameters, simParameters, accretorStar)

    simParameters.dEdrag = dEdrag
    simParameters.dEjet  = dEjet
    simParameters.Edrag = dEdrag * dt

    # Check Binding of the star?
    accretorStar.checkBind(dMdot)
    accretorStar.getAngMom(simParameters)
    
    # inspiral calcualtion
    simParameters.da_dt = a*(dEdrag/np.abs(Eorb))
    simParameters.th_dt = 2.*np.pi/P
    
    return

In [ ]:
mainSimulation(runParameters, simParameters, donorStar, accretorStar, pyMesaContainer, donorModData)

Mesa Called...            14.192123889923096 [secs]
Iteration:                |          0
SaveNum:                  |          0
Time                 [yr] |        0.0
Donor Mass         [Msun] | 10.42016836447483
Accretor Mas       [Msun] | 1.4001536798245895
Donors Max Radius  [Rsun] | 605.1947628812802
Accretor Position  [Rsun] | 422.4648702512745
Wall Clock                |        0.0
Mesa Called...            15.477481842041016 [secs]
Mesa Called...            12.33202075958252 [secs]
Mesa Called...            9.053221225738525 [secs]
Mesa Called...            11.956073760986328 [secs]
Mesa Called...            8.926179647445679 [secs]
Mesa Called...            11.350669622421265 [secs]


In [ ]:
# I think this is supposed to save the inlist and runtime parameters to teh save dir.
ih.copyFile(runParameters.inspiralPath + '/InspiralRuntimeParameters', 
            runParameters.pathAccretor, 
            'InspiralRuntimeParameters.txt')
ih.copyFile(runParameters.inspiralPath + '/inlist_project',
            runParameters.pathAccretor, 
            'inlist_project.txt')